# The Boto–Romão–Silva BFB method, understood by computing it

**A research notebook, not a tutorial.** Notebook 01 established that [DasDey14]'s
boundedness conditions are necessary but not sufficient, and derived the *exact* condition
on the real neutral slice,
$f(t)=\lambda_8t^4+(\lambda_5+\lambda_6+2\lambda_7)t^2-2|\lambda_4|t+(\lambda_1+\lambda_3)\ge0$.
It also said plainly what that leaves open: complex-neutral and charged directions are
covered only by numerical sampling.

[BotoRomaoSilva22] is the methodological model for closing that gap. It is **cited but not
used** anywhere in this project — `constraints.py` implements none of it. This notebook
reads the method and *executes* it, so that what transfers to $S_3$ and what does not is
established by computation rather than by reading.

## The question

Their strategy and ours approximate from **opposite directions**:

| | scope | strength |
|---|---|---|
| our $f(t)\ge0$ (notebook 01) | real neutral slice only | **exact** there — necessary and sufficient on that slice |
| [BotoRomaoSilva22] | all directions, neutral **and** charged | **sufficient** only — a lower bound, so it may reject good points |

Neither contains the other. The useful question is not "which is right" but **which parts
of their machinery our potential can accept**, and that is what §5 settles.

> **Kernel.** Executed with the plain `python3` kernel, not the repo's `lagrangian` kernel,
> which has no `numpy`/`matplotlib`. See `NOTES.md`.


In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import sympy as sp

sys.path.insert(0, str(Path.cwd()))
from model import (build_model, component_symbols, numeric_bfb_min,
                   quartic_potential)
from constraints import bfb_mask, neutral_real_bfb_min, strict_bfb_mask
import derive as dv

sp.init_printing(use_latex="mathjax")
ok = dv.ok
ledger = dv.Ledger("04 — BFB conditions: the Boto–Romão–Silva method")

t0 = time.time()
m = build_model()
lam = m.lam_symbols                       # our lambda_1 ... lambda_8
print("model built in %.1f s" % (time.time() - t0))


model built in 1.0 s


---
## 0. A notation collision that must be settled first

Notebook 01 §4 spent a whole section on the fact that three parametrisations of *our*
potential are in play and must not be conflated. There is now a fourth, and it collides
harder than any of them.

[BotoRomaoSilva22] write the **general** (unconstrained) 3HDM rephasing-invariant quartic
with nine couplings, and they call them $\lambda_1\ldots\lambda_9$:

$$V_{4,\rm RI}=\lambda_1(\phi_1^\dagger\phi_1)^2+\lambda_2(\phi_2^\dagger\phi_2)^2
+\lambda_3(\phi_3^\dagger\phi_3)^2
+\lambda_4(\phi_1^\dagger\phi_1)(\phi_2^\dagger\phi_2)+\ldots
+\lambda_7(\phi_1^\dagger\phi_2)(\phi_2^\dagger\phi_1)+\ldots$$

So **their $\lambda_4$ is $(\phi_1^\dagger\phi_1)(\phi_2^\dagger\phi_2)$, while ours is the
$x_S\!\cdot\!d_2$ structure.** Same symbol, unrelated objects. Every $\lambda_k$ in this
notebook is **ours** unless written $\Lambda_k$, which is theirs. Their derived quantities
keep their own names: $\lambda'_{ij}$, $\hat\lambda_{ij}$, $\bar\lambda_{ij}$.


---
## 1. The decomposition: $V_4=V_N+V_{CB}+V_G$

Their organising move. Write everything in terms of the doublet bilinears
$x_{ij}\equiv\phi_i^\dagger\phi_j$, and set $r_i\equiv x_{ii}=|\phi_i|^2\ge0$. Then

- $V_N=\tfrac12\sum_{ij}r_i A_{ij}r_j$ — a **quadratic form in the norms**, which is all
  that survives along neutral directions;
- $V_{CB}=\sum_{i<j}\lambda'_{ij}z_{ij}$ with
  $$z_{ij}\equiv(\phi_i^\dagger\phi_i)(\phi_j^\dagger\phi_j)-(\phi_i^\dagger\phi_j)(\phi_j^\dagger\phi_i)
  = r_ir_j-|x_{ij}|^2,$$
  the piece that only charge-breaking directions can see;
- $V_G$ — whatever the symmetry leaves over. It **vanishes for U(1)×U(1)**, which is why
  that case is solvable exactly.

Everything downstream rests on one inequality, their Eq. (7):

$$0\;\le\;z_{ij}\;\le\;r_ir_j .$$

> **Before running the next cell.** Both halves are worth deriving before you check them.
> $z_{ij}\ge0$ is Cauchy–Schwarz on two 2-component complex vectors — which one, and in
> which direction? And $z_{ij}\le r_ir_j$ is simply $|x_{ij}|^2\ge0$. Ask yourself which of
> the two is the *load-bearing* one for a **lower** bound on the potential.


In [ ]:
# ---- MOVE 1 + 4: set up the bilinears, then test Eq. (7) numerically. -----
rng = np.random.default_rng(4)
N = 200_000
# random pairs of complex SU(2) doublets
p1 = rng.normal(size=(N, 2)) + 1j*rng.normal(size=(N, 2))
p2 = rng.normal(size=(N, 2)) + 1j*rng.normal(size=(N, 2))

r1 = np.einsum('ij,ij->i', p1.conj(), p1).real
r2 = np.einsum('ij,ij->i', p2.conj(), p2).real
x12 = np.einsum('ij,ij->i', p1.conj(), p2)
z12 = r1*r2 - np.abs(x12)**2

print("over %d random doublet pairs:" % N)
print("   min z_12            = %+.3e" % z12.min())
print("   min (r1 r2 - z_12)  = %+.3e   [ = min |x_12|^2 ]" % (r1*r2 - z12).min())
assert z12.min() >= -1e-9 and (r1*r2 - z12).min() >= -1e-9
ok("0 <= z_ij <= r_i r_j holds — the left half is Cauchy-Schwarz "
   "(|<p1,p2>| <= |p1||p2|), the right half is just |x_ij|^2 >= 0")

# which half does a LOWER bound on V need?  z >= 0, so a term lambda' z is
# bounded below by r_i r_j min(0, lambda') -- see section 3.
ok("z >= 0 is the load-bearing half: it is what lets lambda'_ij z_ij be bounded "
   "below by r_i r_j min(0, lambda'_ij)")


over 200000 random doublet pairs:
   min z_12            = +5.232e-06
   min (r1 r2 - z_12)  = +4.878e-06   [ = min |x_12|^2 ]
✓ 0 <= z_ij <= r_i r_j holds — the left half is Cauchy-Schwarz (|<p1,p2>| <= |p1||p2|), the right half is just |x_ij|^2 >= 0
✓ z >= 0 is the load-bearing half: it is what lets lambda'_ij z_ij be bounded below by r_i r_j min(0, lambda'_ij)


---
## 2. BFB-n is copositivity

Along neutral directions only $V_N=\tfrac12 r^{\mathsf T}\!A\,r$ survives, and the $r_i$ are
**non-negative**. So $V_N\ge0$ is not positive semi-definiteness of $A$ — it is the weaker
condition that $A$ be **copositive**: $r^{\mathsf T}\!A\,r\ge0$ for all $r\ge0$.

For a symmetric $3\times3$ matrix there is a closed form. With $\bar
a_{ij}=a_{ij}+\sqrt{a_{ii}a_{jj}}$, copositivity holds iff

$$a_{ii}\ge0,\qquad \bar a_{ij}\ge0,\qquad
\sqrt{a_{11}a_{22}a_{33}}+a_{12}\sqrt{a_{33}}+a_{13}\sqrt{a_{22}}+a_{23}\sqrt{a_{11}}
+\sqrt{2\,\bar a_{12}\bar a_{13}\bar a_{23}}\;\ge\;0 .$$

> **Before running the next cells.** A positive semi-definite matrix is certainly
> copositive; is the converse true? Construct a $2\times2$ counterexample in your head
> before looking. Then, when we test the closed form against brute-force sampling over
> random directions, predict **which one will be wrong when they disagree** — and what kind
> of matrix will trip the loser.


In [ ]:
# ---- MOVE 1: the closed form, and a direct sampler to test it against. ----
def copositive(A):
    # Closed-form copositivity test for a symmetric 3x3 matrix.
    a, b, c = A[0, 0], A[1, 1], A[2, 2]
    if min(a, b, c) < 0:
        return False
    ab = A[0, 1] + np.sqrt(a*b)
    ac = A[0, 2] + np.sqrt(a*c)
    bc = A[1, 2] + np.sqrt(b*c)
    if min(ab, ac, bc) < 0:
        return False
    return (np.sqrt(a*b*c) + A[0, 1]*np.sqrt(c) + A[0, 2]*np.sqrt(b)
            + A[1, 2]*np.sqrt(a) + np.sqrt(2*ab*ac*bc)) >= 0


def sampled_min(A, n, seed=0):
    # min r^T A r over the unit sphere intersected with the first octant.
    g = np.random.default_rng(seed)
    r = np.abs(g.normal(size=(n, 3)))
    r /= np.linalg.norm(r, axis=1, keepdims=True)
    return np.einsum('ij,jk,ik->i', r, A, r).min()


# psd => copositive, but not conversely: an all-positive matrix is copositive
# whatever its eigenvalues do on the negative orthant.
Apos = np.array([[1.0, -3.0], [-3.0, 1.0]])
print("[[1,-3],[-3,1]] eigenvalues:", np.linalg.eigvalsh(Apos))
Acop = np.array([[1.0, 3.0], [3.0, 1.0]])
print("[[1, 3],[ 3,1]] eigenvalues:", np.linalg.eigvalsh(Acop),
      "  <- indefinite, yet r^T A r >= 0 for r >= 0")
ok("copositive is strictly weaker than positive semi-definite — which is exactly "
   "why BFB-n is not an eigenvalue condition")


[[1,-3],[-3,1]] eigenvalues: [-2.  4.]
[[1, 3],[ 3,1]] eigenvalues: [-2.  4.]   <- indefinite, yet r^T A r >= 0 for r >= 0
✓ copositive is strictly weaker than positive semi-definite — which is exactly why BFB-n is not an eigenvalue condition


In [ ]:
# ---- MOVE 4: closed form vs sampling, over many random matrices. ---------
g = np.random.default_rng(0)
mats = []
for _ in range(4000):
    M = g.uniform(-2, 2, size=(3, 3))
    mats.append((M + M.T)/2)

dis = [A for A in mats if copositive(A) != (sampled_min(A, 20_000, seed=1) >= -1e-9)]
print("4000 random symmetric matrices, closed form vs 20k-direction sampling")
print("   disagreements: %d" % len(dis))


4000 random symmetric matrices, closed form vs 20k-direction sampling
   disagreements: 1


One disagreement — and the interesting question is which side is wrong.


In [ ]:
A = dis[0]
dv.show("the disagreeing matrix", sp.Matrix(np.round(A, 4)), count_terms=False)
print("closed form says copositive :", copositive(A))
for n in (20_000, 500_000, 5_000_000):
    print("   sampled min over %9d directions: %+.4e" % (n, sampled_min(A, n, seed=7)))
print("\n   A[2,2] = %+.4f" % A[2, 2])
print("   r = (0,0,1) gives r^T A r = %+.4f" % A[2, 2])

assert A[2, 2] < 0 and not copositive(A)
ok("the CLOSED FORM is right and the sampling is wrong: A[2,2] < 0, so the pure "
   "e_3 axis already violates copositivity — but 20k random directions never land "
   "close enough to an axis to see it")

dv.falsify("the sampled test can pass a non-copositive matrix",
           lambda: sampled_min(A, 20_000, seed=1) < -1e-9)

ledger.step(
    r"BFB-n is copositivity of $A$, and the closed-form $3\times3$ test is reliable "
    r"where direction sampling is not",
    obtained="derived",
    checks=(r"copositive shown strictly weaker than positive semi-definite",
            r"closed form agrees with sampling on 3999 of 4000 random matrices",
            r"on the one disagreement the closed form is right — the matrix has "
            r"$A_{33}<0$ and is violated only near the $e_3$ axis"),
    falsified_by=r"a matrix where the closed form passes but an exhibited "
                 r"$r\ge0$ gives $r^{\mathsf T}Ar<0$",
    section="2")


the disagreeing matrix


⎡0.7704  0.3765  0.7862 ⎤
⎢                       ⎥
⎢0.3765  1.5124   1.098 ⎥
⎢                       ⎥
⎣0.7862  1.098   -0.0145⎦

closed form says copositive : False
   sampled min over     20000 directions: +1.3433e-02
   sampled min over    500000 directions: -1.1516e-02


   sampled min over   5000000 directions: -1.2894e-02

   A[2,2] = -0.0145
   r = (0,0,1) gives r^T A r = -0.0145
✓ the CLOSED FORM is right and the sampling is wrong: A[2,2] < 0, so the pure e_3 axis already violates copositivity — but 20k random directions never land close enough to an axis to see it
✓ the sampled test can pass a non-copositive matrix — the check fires on a broken input, so it has teeth


Step(claim='BFB-n is copositivity of $A$, and the closed-form $3\\times3$ test is reliable where direction sampling is not', obtained='derived', checks=('copositive shown strictly weaker than positive semi-definite', 'closed form agrees with sampling on 3999 of 4000 random matrices', 'on the one disagreement the closed form is right — the matrix has $A_{33}<0$ and is violated only near the $e_3$ axis'), falsified_by='a matrix where the closed form passes but an exhibited $r\\ge0$ gives $r^{\\mathsf T}Ar<0$', expr=None, section='2', statement=None)

This is the same lesson as notebook 01, arriving from the other side. There, a published
condition tested $f(t)$ at the single point $t=1$ and missed the dip elsewhere. Here,
random sampling covers the interior well and misses the **boundary** — and for copositivity
the boundary is exactly where the violations live, because $r_i\ge0$ makes the axes
admissible directions.

**Both failures are the same failure**: checking a subset of directions and reporting a
verdict about all of them.


---
## 3. The algorithm: sufficient conditions from a lower bound

Exact necessary-and-sufficient BFB conditions are known for U(1)×U(1) (Faro–Ivanov: three
further copositivity tests plus a tetrahedron condition) but are **unknown for Z₂×Z₂**.
[BotoRomaoSilva22]'s contribution is a strategy that works anyway: build a potential
$V^{\rm lower}\le V_4$ that *is* tractable, and test that instead. If the lower bound is
bounded below, so is $V_4$ — hence **sufficient, not necessary**.

The whole method in five steps:

> **ALGORITHM (Boto–Romão–Silva, sufficient BFB)**
>
> 1. **Keep** $V_N$ — it is already a quadratic form in $r$.
> 2. **Bound $V_{CB}$**, using $z_{ij}\ge0$:
>    $\;V_{CB}\;\ge\;\sum_{i<j} r_ir_j\,\min(0,\lambda'_{ij})$.
> 3. **Bound $V_G$.** For the U(1)×Z₂ term,
>    $(\phi_2^\dagger\phi_3)^2+\text{h.c.}\ge-2r_2r_3$, so
>    $V_G\ge-|\bar\lambda_{23}|\,r_2r_3$.
> 4. **Assemble** the shifted matrix
>    $\;\hat\lambda_{ij}=\lambda_{ij}+\min(0,\lambda'_{ij})-|\bar\lambda_{ij}|$.
> 5. **Test copositivity** of that single $3\times3$ $\hat\lambda$.

Steps 2 and 3 are where the exactness is spent: each replaces a field-dependent quantity by
its worst case, which is attained only if some direction realises *all* the worst cases at
once — generally none does.

> **Before running the next cell.** Step 2 replaces $z_{ij}$ by $0$ when $\lambda'_{ij}<0$
> and by... what, when $\lambda'_{ij}>0$? Write out $\min(0,\lambda'_{ij})$ for both signs
> and convince yourself the formula is right in each. Then predict: for which sign of
> $\lambda'_{ij}$ is the bound **tight**, i.e. attained by an actual field configuration?


In [ ]:
# ---- MOVE 1: the algorithm, as code. -------------------------------------
def brs_lower_matrix(lam_N, lam_prime, lam_bar):
    # Steps 1-4: the shifted matrix whose copositivity is the sufficient test.
    #   lam_N     : (3,3) symmetric, the neutral form  V_N = r^T lam_N r
    #   lam_prime : {(i,j): lambda'_ij}    the z_ij coefficients   (step 2)
    #   lam_bar   : {(i,j): lambdabar_ij}  the V_G coefficients    (step 3)
    A = np.array(lam_N, dtype=float).copy()
    for (i, j), lp in lam_prime.items():
        A[i, j] += min(0.0, lp)/2      # /2: the off-diagonal appears twice in r^T A r
        A[j, i] = A[i, j]
    for (i, j), lb in lam_bar.items():
        A[i, j] -= abs(lb)/2
        A[j, i] = A[i, j]
    return A


def brs_sufficient(lam_N, lam_prime, lam_bar):
    # Step 5.  True => the potential IS bounded below (sufficient, not necessary).
    return copositive(brs_lower_matrix(lam_N, lam_prime, lam_bar))


print("step 2, both signs of lambda':")
for lp in (+0.8, -0.8):
    print("   lambda' = %+.1f  ->  min(0, lambda') = %+.1f   "
          "(z in [0, r_i r_j], so the worst case is z=%s)"
          % (lp, min(0.0, lp), "r_i r_j" if lp < 0 else "0"))
ok("for lambda' < 0 the worst case is z = r_i r_j (fully charge-breaking); for "
   "lambda' > 0 it is z = 0 (neutral), and the bound contributes nothing")


step 2, both signs of lambda':
   lambda' = +0.8  ->  min(0, lambda') = +0.0   (z in [0, r_i r_j], so the worst case is z=0)
   lambda' = -0.8  ->  min(0, lambda') = -0.8   (z in [0, r_i r_j], so the worst case is z=r_i r_j)
✓ for lambda' < 0 the worst case is z = r_i r_j (fully charge-breaking); for lambda' > 0 it is z = 0 (neutral), and the bound contributes nothing


So step 2 is tight for **either** sign taken alone — $z_{ij}$ really does reach both ends of
$[0,r_ir_j]$. The looseness enters only when several bounds must be saturated
*simultaneously* by one field configuration. Let us measure that directly.


In [ ]:
# ---- MOVE 4: is it actually a lower bound, and how loose? ---------------
# A toy 3HDM piece we can evaluate exactly: V = r^T lam_N r + lambda'_12 z_12.
gg = np.random.default_rng(11)
K = 100_000
q1 = gg.normal(size=(K, 2)) + 1j*gg.normal(size=(K, 2))
q2 = gg.normal(size=(K, 2)) + 1j*gg.normal(size=(K, 2))
q3 = gg.normal(size=(K, 2)) + 1j*gg.normal(size=(K, 2))
R = np.stack([np.einsum('ij,ij->i', q.conj(), q).real for q in (q1, q2, q3)], axis=1)
Z12 = R[:, 0]*R[:, 1] - np.abs(np.einsum('ij,ij->i', q1.conj(), q2))**2

lam_N = np.array([[1.0, 0.3, -0.2], [0.3, 0.8, 0.1], [-0.2, 0.1, 1.2]])
lp12 = -0.7

V_true = np.einsum('ij,jk,ik->i', R, lam_N, R) + lp12*Z12
V_low = np.einsum('ij,jk,ik->i', R,
                  brs_lower_matrix(lam_N, {(0, 1): lp12}, {}), R)

gap = V_true - V_low
print("over %d random field configurations:" % K)
print("   min (V_true - V_lower) = %+.4e   <- must be >= 0" % gap.min())
print("   median gap             = %+.4e" % np.median(gap))
assert gap.min() >= -1e-9
ok("V_lower is a genuine lower bound on every configuration sampled")

# and the bound is attained: some configuration saturates it
print("\n   configurations within 1e-3 of saturating it: %d"
      % int((gap < 1e-3 * np.abs(V_true).mean()).sum()))

dv.falsify("the lower bound depends on the sign convention in step 2",
           lambda: (np.einsum('ij,jk,ik->i', R, lam_N, R) + lp12*Z12
                    - np.einsum('ij,jk,ik->i', R, brs_lower_matrix(
                        lam_N, {(0, 1): -lp12}, {}), R)).min() >= -1e-9)

ledger.step(
    r"The lower-bound strategy: $V_4\ge V_N+V_{CB}^{\rm lower}+V_G^{\rm lower}$, "
    r"reducing BFB to one $3\times3$ copositivity test",
    obtained="imported",
    statement=r"\hat\lambda_{ij}=\lambda_{ij}+\min(0,\lambda'_{ij})-|\bar\lambda_{ij}|,"
              r"\qquad \text{BFB} \Longleftarrow \hat\lambda \text{ copositive}",
    checks=(r"$V_{\rm lower}\le V_4$ verified on $10^5$ random field configurations",
            r"the bound is attained, so step 2 is tight for either sign alone",
            r"flipping the sign in $\min(0,\lambda')$ breaks it"),
    falsified_by=r"a field configuration with $V_4<V_{\rm lower}$",
    section="3 (from [BotoRomaoSilva22])")


over 100000 random field configurations:
   min (V_true - V_lower) = +7.7993e-05   <- must be >= 0
   median gap             = +2.8810e+00
✓ V_lower is a genuine lower bound on every configuration sampled

   configurations within 1e-3 of saturating it: 2565
✓ the lower bound depends on the sign convention in step 2 — the check fires on a broken input, so it has teeth


Step(claim='The lower-bound strategy: $V_4\\ge V_N+V_{CB}^{\\rm lower}+V_G^{\\rm lower}$, reducing BFB to one $3\\times3$ copositivity test', obtained='imported', checks=('$V_{\\rm lower}\\le V_4$ verified on $10^5$ random field configurations', 'the bound is attained, so step 2 is tight for either sign alone', "flipping the sign in $\\min(0,\\lambda')$ breaks it"), falsified_by='a field configuration with $V_4<V_{\\rm lower}$', expr=None, section='3 (from [BotoRomaoSilva22])', statement="\\hat\\lambda_{ij}=\\lambda_{ij}+\\min(0,\\lambda'_{ij})-|\\bar\\lambda_{ij}|,\\qquad \\text{BFB} \\Longleftarrow \\hat\\lambda \\text{ copositive}")

---
## 4. Applying the decomposition to **our** $S_3$ potential

Now the part that is ours, not theirs. [BotoRomaoSilva22] treat U(1)×U(1), U(1)×Z₂ and
Z₂×Z₂. $S_3$ is not among them, so nothing can be quoted — the decomposition has to be
*done*.

Our quartic potential, in the $x_{ij}$ notation of notebook 01:

$$V_4=\lambda_1(x_{11}+x_{22})^2+\lambda_2(x_{12}-x_{21})^2+\lambda_3|d_2|^2
+\lambda_4\big[x_S\!\cdot\!d_2+\text{h.c.}\big]
+\lambda_5x_{SS}(x_{11}+x_{22})+\lambda_6\big[x_{S1}x_{1S}+x_{S2}x_{2S}\big]
+\lambda_7\big[x_{S1}^2+x_{S2}^2+\text{h.c.}\big]+\lambda_8x_{SS}^2 .$$

> **Before running the next cell.** Sort our eight couplings into their three bins by
> inspection first. Which are manifestly quadratic in the norms $r_i$ alone ($V_N$)? Which
> produce $|x_{ij}|^2$, hence $z_{ij}$ ($V_{CB}$)? Which produce a *squared* off-diagonal
> bilinear $x_{ij}^2+\text{h.c.}$, the shape of their U(1)×Z₂ term ($V_G$)? And — the one
> that matters — **is there any term left over that fits none of the three?**


In [ ]:
# ---- MOVE 1: build V4 symbolically, and abstract symbols for the bilinears.
V = -sum(t.expr for t in m.model.lagrangian.terms)
V4 = sum(t for t in sp.expand(V).args if any(t.has(x) for x in lam))

H = [list(d.components) for d in m.doublets]
bil = lambda i, j: sum(sp.conjugate(H[i][a])*H[j][a] for a in range(2))
r = [bil(i, i) for i in range(3)]
x12, x1S, x2S = bil(0, 1), bil(0, 2), bil(1, 2)
l = {k: lam[k-1] for k in range(1, 9)}

# abstract stand-ins: R_i for the norms, (a,b) for Re/Im of each off-diagonal
R1, R2, RS = sp.symbols("R_1 R_2 R_S", nonnegative=True)
a12, b12, a1S, b1S, a2S, b2S = sp.symbols("a12 b12 a1S b1S a2S b2S", real=True)
PAIR = {"12": (a12, b12), "1S": (a1S, b1S), "2S": (a2S, b2S)}
print("norms R_1,R_2,R_S  and  Re/Im of x_12, x_1S, x_2S")


# ---- MOVE 2: collect V4 into their three bins, plus a remainder. --------
V_N = (l[1]*(R1 + R2)**2 + l[3]*(R1 - R2)**2
       + l[5]*RS*(R1 + R2) + l[8]*RS**2)
V_CB = (2*(l[3] - l[2])*(a12**2 + b12**2)
        + l[6]*(a1S**2 + b1S**2 + a2S**2 + b2S**2))
V_G = (2*(l[3] + l[2])*(a12**2 - b12**2)
       + 2*l[7]*((a1S**2 - b1S**2) + (a2S**2 - b2S**2)))
V_rest = 2*l[4]*(a1S*(R1 - R2) - 2*a2S*a12)

back = {R1: r[0], R2: r[1], RS: r[2],
        a12: (x12 + sp.conjugate(x12))/2, b12: (x12 - sp.conjugate(x12))/(2*sp.I),
        a1S: (x1S + sp.conjugate(x1S))/2, b1S: (x1S - sp.conjugate(x1S))/(2*sp.I),
        a2S: (x2S + sp.conjugate(x2S))/2, b2S: (x2S - sp.conjugate(x2S))/(2*sp.I)}
residual = sp.simplify(sp.expand((V_N + V_CB + V_G + V_rest).subs(back) - V4))
print("\nV_4 - (V_N + V_CB + V_G + V_rest) =", residual)
assert residual == 0
ok("the decomposition is EXACT — an identity in the field components, so everything "
   "below rests on algebra rather than on inspection")


norms R_1,R_2,R_S  and  Re/Im of x_12, x_1S, x_2S

V_4 - (V_N + V_CB + V_G + V_rest) = 0
✓ the decomposition is EXACT — an identity in the field components, so everything below rests on algebra rather than on inspection


So the split is exact, and it reads:

$$
\begin{aligned}
V_N &= \lambda_1(r_1+r_2)^2+\lambda_3(r_1-r_2)^2+\lambda_5r_S(r_1+r_2)+\lambda_8r_S^2,\\[2pt]
V_{CB} &= 2(\lambda_3-\lambda_2)|x_{12}|^2+\lambda_6\big(|x_{1S}|^2+|x_{2S}|^2\big),\\[2pt]
V_G &= 2(\lambda_3+\lambda_2)\,\mathrm{Re}\,x_{12}^2
      +2\lambda_7\big(\mathrm{Re}\,x_{1S}^2+\mathrm{Re}\,x_{2S}^2\big),\\[2pt]
V_{\rm rest} &= 2\lambda_4\big[\mathrm{Re}(x_{1S})(r_1-r_2)-2\,\mathrm{Re}(x_{2S})\,\mathrm{Re}(x_{12})\big].
\end{aligned}
$$

Seven of our eight couplings land cleanly. $\lambda_1,\lambda_5,\lambda_8$ and part of
$\lambda_3$ go to $V_N$; $\lambda_6$ and the rest of $\lambda_2,\lambda_3$ to $V_{CB}$ via
$|x_{ij}|^2=r_ir_j-z_{ij}$; and $\lambda_7$ together with the $\mathrm{Re}\,x_{12}^2$
combination to $V_G$ — **exactly the shape of their U(1)×Z₂ term**
$(\phi_2^\dagger\phi_3)^2+\text{h.c.}$, so their step-3 bound applies verbatim.

And then there is $\lambda_4$. To say precisely what is different about it, classify every
monomial by *which* off-diagonal bilinears it contains and to what degree.


In [ ]:
# ---- MOVE 3: recognise.  Classify monomials by their off-diagonal content.
def signatures(expr):
    # for each monomial: {pair: total degree in that pair's Re/Im symbols}
    out = set()
    for mono in sp.Add.make_args(sp.expand(expr)):
        d = {}
        for name, (aa, bb) in PAIR.items():
            deg = sp.degree(mono, aa) + sp.degree(mono, bb)
            if deg:
                d[name] = deg
        out.add(tuple(sorted(d.items())))
    return sorted(out)


for name, part in (("V_N", V_N), ("V_CB", V_CB), ("V_G", V_G), ("V_rest", V_rest)):
    print("%-8s %s" % (name, signatures(part) or [()]))

theirs = set(signatures(V_CB)) | set(signatures(V_G)) | set(signatures(V_N))
ours = set(signatures(V_rest))
print("\nshapes their framework produces :", sorted(theirs))
print("shapes V_rest produces          :", sorted(ours))
print("shapes with NO analogue         :", sorted(ours - theirs))
assert not (ours & theirs)
ok("every V_CB / V_G monomial is degree 2 in ONE pair; V_rest's are degree 1 in one "
   "pair, or degree 1 in EACH of two different pairs — neither shape occurs anywhere "
   "in their framework")


V_N      [()]
V_CB     [(('12', 2),), (('1S', 2),), (('2S', 2),)]
V_G      [(('12', 2),), (('1S', 2),), (('2S', 2),)]
V_rest   [(('12', 1), ('2S', 1)), (('1S', 1),)]

shapes their framework produces : [(), (('12', 2),), (('1S', 2),), (('2S', 2),)]
shapes V_rest produces          : [(('12', 1), ('2S', 1)), (('1S', 1),)]
shapes with NO analogue         : [(('12', 1), ('2S', 1)), (('1S', 1),)]
✓ every V_CB / V_G monomial is degree 2 in ONE pair; V_rest's are degree 1 in one pair, or degree 1 in EACH of two different pairs — neither shape occurs anywhere in their framework


---
## 5. Why $\lambda_4$ resists — and where the obstruction reappears

Their whole framework produces only two monomial shapes: no off-diagonal bilinears at all
($V_N$), or **degree 2 in a single pair** ($|x_{ij}|^2$ and $\mathrm{Re}\,x_{ij}^2$). That is
what makes $V_4$ a quadratic form in $r$ once the bounds are applied, and hence what makes
copositivity the right test.

$V_{\rm rest}$ produces neither shape. Bound it the only way available —
$|\mathrm{Re}\,x_{ij}|\le|x_{ij}|\le\sqrt{r_ir_j}$ — and watch what happens.

> **Before running the next cell.** Apply that bound to both monomials of $V_{\rm rest}$ on
> paper. The result is still degree 2 in the $r_i$, as it must be. So why can it *not* be
> folded into $\hat\lambda$? Look at what the square roots do to the *powers*, and recall
> which variable notebook 01 had to introduce to make its own condition polynomial.


In [ ]:
# ---- MOVE 1 + 2: bound V_rest, in s_i = sqrt(r_i). ----------------------
s1, s2, sS = sp.symbols("s_1 s_2 s_S", nonnegative=True)
L4 = sp.Symbol("|\\lambda_{4}|", nonnegative=True)

# |Re x_1S| <= s_1 s_S ,  |Re x_2S| <= s_2 s_S ,  |Re x_12| <= s_1 s_2
worst = -2*L4*(s1*sS*(s1**2 + s2**2) + 2*(s2*sS)*(s1*s2))
dv.show("worst case of V_rest, written in s_i = sqrt(r_i)", sp.expand(worst))

P = sp.Poly(sp.expand(worst), s1, s2, sS)
deg_s = P.total_degree()
# a polynomial in r_i = s_i^2 is exactly one whose every EXPONENT is even.
odd = [mo for mo in P.monoms() if any(e % 2 for e in mo)]
print("\ndegree in s_i = sqrt(r_i) :", deg_s)
print("monomial exponents (s_1, s_2, s_S):", P.monoms())
print("monomials with an ODD exponent    :", odd, " -> not a polynomial in r_i")
assert deg_s == 4 and odd
ok("the bound is a QUARTIC form in s_i = sqrt(r_i) whose exponents are odd, so it is "
   "NOT a polynomial in r_i at all — the 3x3 copositivity test has nothing to bite on")


worst case of V_rest, written in s_i = sqrt(r_i)   [2 terms]


      3                            2                  
- 2⋅s₁ ⋅s_S⋅|\lambda_{4}| - 6⋅s₁⋅s₂ ⋅s_S⋅|\lambda_{4}|


degree in s_i = sqrt(r_i) : 4
monomial exponents (s_1, s_2, s_S): [(3, 0, 1), (1, 2, 1)]
monomials with an ODD exponent    : [(3, 0, 1), (1, 2, 1)]  -> not a polynomial in r_i
✓ the bound is a QUARTIC form in s_i = sqrt(r_i) whose exponents are odd, so it is NOT a polynomial in r_i at all — the 3x3 copositivity test has nothing to bite on


There it is. Bounding $\lambda_4$ forces odd powers of $\sqrt{r_i}$, so the object to be
tested is a **quartic form in $s_i=\sqrt{r_i}$** rather than a quadratic form in $r_i$.

This is exactly the obstruction notebook 01 hit from the other direction. There, restricting
to the real neutral slice gave

$$V_4=(\lambda_1+\lambda_3)x^2+(\lambda_5+\lambda_6+2\lambda_7)xy+\lambda_8y^2
+2\lambda_4\sqrt y\,x^{3/2}\cos3\psi,$$

with the same half-integer powers, and the fix was to set $x=1$, $t=\sqrt y$ and minimise a
**quartic in $t$**. Same substitution, same cause, reached from the opposite side. So the
honest statement is not "their method fails for $S_3$" but this:


In [ ]:
ledger.step(
    r"Our $V_4$ decomposes exactly into their $V_N+V_{CB}+V_G$ **plus** a remainder "
    r"$\propto\lambda_4$ whose monomial shapes occur nowhere in their framework",
    obtained="derived",
    statement=r"V_{\rm rest}=2\lambda_4\big[\mathrm{Re}(x_{1S})(r_1-r_2)"
              r"-2\,\mathrm{Re}(x_{2S})\,\mathrm{Re}(x_{12})\big]",
    checks=(r"the four-way split reproduces $V_4$ exactly (residual $=0$ symbolically)",
            r"$V_{CB},V_G$ monomials are degree 2 in one bilinear pair; "
            r"$V_{\rm rest}$'s are degree 1 in one pair, or in each of two pairs",
            r"the two shape sets are disjoint",
            r"bounding $V_{\rm rest}$ gives a quartic form in $\sqrt{r_i}$ with odd "
            r"powers — the same obstruction as notebook 01's $t=\sqrt y$"),
    falsified_by=r"a rewriting of $V_{\rm rest}$ using only degree-2-in-one-pair "
                 r"monomials, which would put it inside their framework after all",
    section="4-5")


Step(claim='Our $V_4$ decomposes exactly into their $V_N+V_{CB}+V_G$ **plus** a remainder $\\propto\\lambda_4$ whose monomial shapes occur nowhere in their framework', obtained='derived', checks=('the four-way split reproduces $V_4$ exactly (residual $=0$ symbolically)', "$V_{CB},V_G$ monomials are degree 2 in one bilinear pair; $V_{\\rm rest}$'s are degree 1 in one pair, or in each of two pairs", 'the two shape sets are disjoint', "bounding $V_{\\rm rest}$ gives a quartic form in $\\sqrt{r_i}$ with odd powers — the same obstruction as notebook 01's $t=\\sqrt y$"), falsified_by='a rewriting of $V_{\\rm rest}$ using only degree-2-in-one-pair monomials, which would put it inside their framework after all', expr=None, section='4-5', statement='V_{\\rm rest}=2\\lambda_4\\big[\\mathrm{Re}(x_{1S})(r_1-r_2)-2\\,\\mathrm{Re}(x_{2S})\\,\\mathrm{Re}(x_{12})\\big]')

---
## 6. Where both methods apply: the $\lambda_4=0$ slice

Set $\lambda_4=0$ and our potential lands entirely inside their framework — the one place
the two approaches can be compared head to head.

Reading the coefficients off §4 with $|x_{ij}|^2=r_ir_j-z_{ij}$:

$$\lambda'_{12}=2(\lambda_2-\lambda_3),\quad \lambda'_{1S}=\lambda'_{2S}=-\lambda_6,
\qquad \bar\lambda_{12}=2(\lambda_2+\lambda_3),\quad
\bar\lambda_{1S}=\bar\lambda_{2S}=2\lambda_7 .$$

> **Before running the next cells.** Their test is *sufficient*; notebook 01's $f(t)\ge0$ is
> *necessary*. On any parameter point, one of (pass, pass), (pass, fail), (fail, pass),
> (fail, fail) is impossible — predict which, and what that says about the logical relation
> between the two.


In [ ]:
# ---- MOVE 1: their hatted matrix, for OUR potential at lambda_4 = 0. ----
def s3_brs_matrix(L):
    # L = (lambda_1 ... lambda_8).  V_N plus the r_i r_j freed by
    # |x_ij|^2 = r_i r_j - z_ij, then steps 2-3 of the algorithm.
    L1, L2, L3, L4, L5, L6, L7, L8 = L
    A = np.zeros((3, 3))
    A[0, 0] = A[1, 1] = L1 + L3
    A[2, 2] = L8
    A[0, 1] = A[1, 0] = L1 - L2                    # 2(L1-L3)/2 + 2(L3-L2)/2
    A[0, 2] = A[2, 0] = (L5 + L6)/2
    A[1, 2] = A[2, 1] = (L5 + L6)/2
    lam_prime = {(0, 1): 2*(L2 - L3), (0, 2): -L6, (1, 2): -L6}
    lam_bar = {(0, 1): 2*(L2 + L3), (0, 2): 2*L7, (1, 2): 2*L7}
    return brs_lower_matrix(A, lam_prime, lam_bar)


# ---- MOVE 4a: is r^T lambdahat r really below V_4?  Check pointwise. ----
gc = np.random.default_rng(31)
KC = 40_000
c1 = gc.normal(size=(KC, 2)) + 1j*gc.normal(size=(KC, 2))
c2 = gc.normal(size=(KC, 2)) + 1j*gc.normal(size=(KC, 2))
c3 = gc.normal(size=(KC, 2)) + 1j*gc.normal(size=(KC, 2))
Rn = np.stack([np.einsum('ij,ij->i', c.conj(), c).real for c in (c1, c2, c3)], axis=1)

V4fn = quartic_potential(m)     # lambdified V_4(x0..x5, y0..y5, lam1..lam8)
worstgap = np.inf
for _ in range(40):
    Lk = gc.uniform(-2, 2, size=8); Lk[3] = 0.0
    # component order is component_symbols(m): H1p,H10,H2p,H20,HSp,HS0
    v_true = V4fn(c1.real[:, 0], c1.real[:, 1], c2.real[:, 0], c2.real[:, 1],
                  c3.real[:, 0], c3.real[:, 1],
                  c1.imag[:, 0], c1.imag[:, 1], c2.imag[:, 0], c2.imag[:, 1],
                  c3.imag[:, 0], c3.imag[:, 1], *Lk)
    v_low = np.einsum('ij,jk,ik->i', Rn, s3_brs_matrix(Lk), Rn)
    worstgap = min(worstgap, (v_true - v_low).min())
print("min over 40 lambda points x %d field configs of (V_4 - r^T lambdahat r): %+.4e"
      % (KC, worstgap))
assert worstgap >= -1e-6
ok("r^T lambdahat r is a genuine lower bound on V_4 at lambda_4 = 0 — which validates "
   "the whole assembly, matrix and steps 2-3 together")


min over 40 lambda points x 40000 field configs of (V_4 - r^T lambdahat r): +7.0491e-02
✓ r^T lambdahat r is a genuine lower bound on V_4 at lambda_4 = 0 — which validates the whole assembly, matrix and steps 2-3 together


In [ ]:
# ---- MOVE 4b: the logical relation between the two tests. ---------------
gs = np.random.default_rng(2026)
NL = 1500
L = gs.uniform(-2, 2, size=(NL, 8))
L[:, 3] = 0.0                                        # lambda_4 = 0

brs_pass = np.array([copositive(s3_brs_matrix(Lk)) for Lk in L])
worst12 = numeric_bfb_min(m, L, n_directions=20_000, seed=5)
bad = int(((worst12 < -1e-6) & brs_pass).sum())
print("lambda_4 = 0 slice, %d points" % NL)
print("   their sufficient test passes            : %d" % brs_pass.sum())
print("   of those, unbounded by a direct 12-d scan: %d" % bad)
assert bad == 0
ok("their sufficient condition never passes a point the direct scan proves unbounded "
   "— which is what 'sufficient' has to mean")

ours_pass = neutral_real_bfb_min(L) >= -1e-9          # notebook 01's f(t) >= 0
tbl = [[int(((brs_pass == a) & (ours_pass == b)).sum()) for b in (True, False)]
       for a in (True, False)]
print("\n                          ours (f(t) >= 0)")
print("                           pass     fail")
print("   theirs (sufficient) pass %6d   %6d" % (tbl[0][0], tbl[0][1]))
print("                       fail %6d   %6d" % (tbl[1][0], tbl[1][1]))
assert tbl[0][1] == 0
ok("the (theirs pass, ours fail) cell is EMPTY, as it must be: theirs implies bounded, "
   "ours is necessary for bounded, so theirs must imply ours")
print("\n   (theirs fail, ours pass) = %d points — theirs is simply too conservative "
      "to decide there, and ours is silent off the real neutral slice" % tbl[1][0])

ledger.step(
    r"On the $\lambda_4=0$ slice the two tests are strictly ordered: their sufficient "
    r"condition $\Rightarrow$ our necessary $f(t)\ge0$, never the reverse",
    obtained="derived",
    checks=(r"$r^{\mathsf T}\hat\lambda r\le V_4$ verified pointwise over 40 $\lambda$ "
            r"points $\times\,4\times10^4$ field configurations",
            r"their test never passes a point a direct 12-dimensional scan proves "
            r"unbounded",
            r"the (theirs pass, ours fail) cell of the $2\times2$ table is empty",
            r"the (theirs fail, ours pass) cell is populated, so neither test is "
            r"equivalent to the other"),
    falsified_by=r"a single point their condition accepts and $f(t)\ge0$ rejects",
    section="6")


lambda_4 = 0 slice, 1500 points
   their sufficient test passes            : 46
   of those, unbounded by a direct 12-d scan: 0
✓ their sufficient condition never passes a point the direct scan proves unbounded — which is what 'sufficient' has to mean

                          ours (f(t) >= 0)
                           pass     fail
   theirs (sufficient) pass     46        0
                       fail    248     1206
✓ the (theirs pass, ours fail) cell is EMPTY, as it must be: theirs implies bounded, ours is necessary for bounded, so theirs must imply ours

   (theirs fail, ours pass) = 248 points — theirs is simply too conservative to decide there, and ours is silent off the real neutral slice


Step(claim='On the $\\lambda_4=0$ slice the two tests are strictly ordered: their sufficient condition $\\Rightarrow$ our necessary $f(t)\\ge0$, never the reverse', obtained='derived', checks=('$r^{\\mathsf T}\\hat\\lambda r\\le V_4$ verified pointwise over 40 $\\lambda$ points $\\times\\,4\\times10^4$ field configurations', 'their test never passes a point a direct 12-dimensional scan proves unbounded', 'the (theirs pass, ours fail) cell of the $2\\times2$ table is empty', 'the (theirs fail, ours pass) cell is populated, so neither test is equivalent to the other'), falsified_by='a single point their condition accepts and $f(t)\\ge0$ rejects', expr=None, section='6', statement=None)

---
## 7. What extending the method to $S_3$ would take

**Open.** This notebook establishes what is missing, not how to supply it. What is now
precise:

1. **The gap is one term.** $V_N$, $V_{CB}$ and $V_G$ of our potential are all inside
   [BotoRomaoSilva22]'s framework; only $V_{\rm rest}\propto\lambda_4$ is not, and it fails
   for a structural reason (linear, not quadratic, in the off-diagonal bilinears).

2. **The obstruction has a name.** Bounding it produces a quartic form in $\sqrt{r_i}$. Any
   extension must either minimise a quartic form on the non-negative orthant — the
   $\sqrt{\ }$ analogue of copositivity — or find a smarter bound that stays quadratic in
   $r$.

3. **We already solve it on one slice.** Notebook 01's $f(t)\ge0$ is exactly the
   one-variable case of that quartic-form problem, solved in closed form. The natural
   construction is a **hybrid**: their machinery for the charged and complex-neutral
   directions, ours for the $\lambda_4$ structure.

4. **A cross-check exists.** [Yildirim26] applies vacuum-stability constraints to this same
   $S_3$ model; whatever conditions it uses are a target to compare against. (The
   [DasDey14] erratum is no longer an unknown here: it corrects only the tadpole
   equations, leaving Eq. (4) untouched — see notebook 01 §2.2.)

None of this is implemented. `constraints.py` is untouched by this notebook — it still
computes [DasDey14] Eq. (4) plus notebook 01's correction, and that is what every number in
`report_scalar_sector.tex` rests on.


---
## 8. The ledger, and the algebra on paper


In [ ]:
ledger.step(
    r"[BotoRomaoSilva22]'s decomposition $V_4=V_N+V_{CB}+V_G$ and the bound "
    r"$0\le z_{ij}\le r_ir_j$",
    obtained="imported",
    checks=(r"$0\le z_{ij}\le r_ir_j$ verified on $2\times10^5$ random doublet pairs",
            r"the left half identified as Cauchy--Schwarz, the right as $|x_{ij}|^2\ge0$"),
    falsified_by=r"a doublet pair with $z_{ij}<0$ or $z_{ij}>r_ir_j$",
    section="1 (from [BotoRomaoSilva22])")
ledger.table()
ledger.to_latex("derivations_04.tex",
                subtitle="the Boto--Romao--Silva BFB method, applied to $S_3$")


### Derivation ledger — 04 — BFB conditions: the Boto–Romão–Silva method

**1. BFB-n is copositivity of $A$, and the closed-form $3\times3$ test is reliable where direction sampling is not**  
<sub>derived here &nbsp;·&nbsp; §2</sub>

- ✓ copositive shown strictly weaker than positive semi-definite
- ✓ closed form agrees with sampling on 3999 of 4000 random matrices
- ✓ on the one disagreement the closed form is right — the matrix has $A_{33}<0$ and is violated only near the $e_3$ axis

*Would be falsified by:* a matrix where the closed form passes but an exhibited $r\ge0$ gives $r^{\mathsf T}Ar<0$


---

**2. The lower-bound strategy: $V_4\ge V_N+V_{CB}^{\rm lower}+V_G^{\rm lower}$, reducing BFB to one $3\times3$ copositivity test**  
<sub>**imported** &nbsp;·&nbsp; §3 (from [BotoRomaoSilva22])</sub>

$$\hat\lambda_{ij}=\lambda_{ij}+\min(0,\lambda'_{ij})-|\bar\lambda_{ij}|,\qquad \text{BFB} \Longleftarrow \hat\lambda \text{ copositive}$$

- ✓ $V_{\rm lower}\le V_4$ verified on $10^5$ random field configurations
- ✓ the bound is attained, so step 2 is tight for either sign alone
- ✓ flipping the sign in $\min(0,\lambda')$ breaks it

*Would be falsified by:* a field configuration with $V_4<V_{\rm lower}$


---

**3. Our $V_4$ decomposes exactly into their $V_N+V_{CB}+V_G$ **plus** a remainder $\propto\lambda_4$ whose monomial shapes occur nowhere in their framework**  
<sub>derived here &nbsp;·&nbsp; §4-5</sub>

$$V_{\rm rest}=2\lambda_4\big[\mathrm{Re}(x_{1S})(r_1-r_2)-2\,\mathrm{Re}(x_{2S})\,\mathrm{Re}(x_{12})\big]$$

- ✓ the four-way split reproduces $V_4$ exactly (residual $=0$ symbolically)
- ✓ $V_{CB},V_G$ monomials are degree 2 in one bilinear pair; $V_{\rm rest}$'s are degree 1 in one pair, or in each of two pairs
- ✓ the two shape sets are disjoint
- ✓ bounding $V_{\rm rest}$ gives a quartic form in $\sqrt{r_i}$ with odd powers — the same obstruction as notebook 01's $t=\sqrt y$

*Would be falsified by:* a rewriting of $V_{\rm rest}$ using only degree-2-in-one-pair monomials, which would put it inside their framework after all


---

**4. On the $\lambda_4=0$ slice the two tests are strictly ordered: their sufficient condition $\Rightarrow$ our necessary $f(t)\ge0$, never the reverse**  
<sub>derived here &nbsp;·&nbsp; §6</sub>

- ✓ $r^{\mathsf T}\hat\lambda r\le V_4$ verified pointwise over 40 $\lambda$ points $\times\,4\times10^4$ field configurations
- ✓ their test never passes a point a direct 12-dimensional scan proves unbounded
- ✓ the (theirs pass, ours fail) cell of the $2\times2$ table is empty
- ✓ the (theirs fail, ours pass) cell is populated, so neither test is equivalent to the other

*Would be falsified by:* a single point their condition accepts and $f(t)\ge0$ rejects


---

**5. [BotoRomaoSilva22]'s decomposition $V_4=V_N+V_{CB}+V_G$ and the bound $0\le z_{ij}\le r_ir_j$**  
<sub>**imported** &nbsp;·&nbsp; §1 (from [BotoRomaoSilva22])</sub>

- ✓ $0\le z_{ij}\le r_ir_j$ verified on $2\times10^5$ random doublet pairs
- ✓ the left half identified as Cauchy--Schwarz, the right as $|x_{ij}|^2\ge0$

*Would be falsified by:* a doublet pair with $z_{ij}<0$ or $z_{ij}>r_ir_j$


---

<sub>5 steps: 3 derived here, 2 imported, 0 assumed</sub>

wrote derivations_04.tex  (5 steps)


'derivations_04.tex'

---
## 9. References

- **[BotoRomaoSilva22]** R. Boto, J. C. Romão, J. P. Silva, *"Bounded from below conditions
  on a class of symmetry constrained 3HDM"*, Phys. Rev. D **106**, 115010 (2022),
  [arXiv:2208.01068](https://arxiv.org/abs/2208.01068),
  [doi:10.1103/PhysRevD.106.115010](https://doi.org/10.1103/PhysRevD.106.115010).
  The method read and executed here: the $V_N/V_{CB}/V_G$ split, $0\le z_{ij}\le r_ir_j$,
  BFB-n as copositivity, and the lower-bound strategy for sufficient conditions. Covers
  U(1)×U(1), U(1)×Z₂, Z₂×Z₂ — **not** S₃.
- **[DasDey14]** D. Das, U. K. Dey, *"Analysis of an extended scalar sector with S₃
  symmetry"*, Phys. Rev. D **89**, 095025 (2014),
  [arXiv:1404.2491](https://arxiv.org/abs/1404.2491),
  [doi:10.1103/PhysRevD.89.095025](https://doi.org/10.1103/PhysRevD.89.095025).
  Erratum: Phys. Rev. D **91**, 039905 (2015) — not on arXiv, but freely readable at APS;
  **obtained**. It corrects Eqs. (9a)–(9c) (the tadpole conditions) and nothing else, so
  Eq. (4) — the source of the conditions `constraints.py` implements — is unaffected. See
  notebook 01 §2.2.
- **[GomezBock21]** M. Gómez-Bock, M. Mondragón, A. Pérez-Martínez, Eur. Phys. J. C **81**,
  942 (2021), [arXiv:2102.02800](https://arxiv.org/abs/2102.02800),
  [doi:10.1140/epjc/s10052-021-09731-3](https://doi.org/10.1140/epjc/s10052-021-09731-3).
- **[Yildirim26]** E. Yildirim, *"Double SM-like Higgs Production at future $e^+e^-$
  colliders in the 3-Higgs Doublet Model under the $S_3$ symmetry"*,
  [arXiv:2604.24421](https://arxiv.org/abs/2604.24421) (2026). Applies vacuum-stability
  constraints to this same model; a cross-check target (§7).

### What this notebook takes from where

| ingredient | source | status here |
|---|---|---|
| $V_4=V_N+V_{CB}+V_G$; $0\le z_{ij}\le r_ir_j$ | [BotoRomaoSilva22] §II | **imported**, verified numerically (§1) |
| BFB-n $=$ copositivity of $A$ | [BotoRomaoSilva22] §III | **imported**, closed form verified vs sampling (§2) |
| the lower-bound strategy, steps 1–5 | [BotoRomaoSilva22] §III–V | **imported**, verified to be a lower bound (§3) |
| the decomposition of **our** $S_3$ $V_4$ | — | **derived here** (§4), exact identity |
| $\lambda_4$ has no analogue in their classes | — | **derived here** (§4–5) |
| the $\sqrt{r_i}$ obstruction | — | **derived here** (§5), matches notebook 01 |
| the $\lambda_4=0$ comparison | — | **derived here** (§6) |
